# 00 — Setup and Preprocessing

Notebook ini menyiapkan lingkungan, membersihkan teks, membangun tiga
representasi HMRNet, dan menyimpan artefak bersama agar notebook lain
tidak mengulang preprocessing.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

PROJECT_BOOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_BOOT.exists():
    PROJECT_BOOT = Path("/content/drive/MyDrive/HMRNet_Experiments")

assert PROJECT_BOOT.exists(), (
    "Folder proyek belum ditemukan. Ekstrak HMRNet_Experiments.zip "
    "ke /content atau /content/drive/MyDrive."
)

%cd {PROJECT_BOOT}
!pip -q install -r requirements.txt


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## Konfigurasi dataset

Ganti `DATA_PATH`. Nama kolom boleh dibiarkan `None` agar dideteksi otomatis.


In [ ]:
DATA_PATH = "/content/drive/MyDrive/UDB/Riset/2026/dataset/dataset_mbg.csv"

TEXT_COLUMN = "full_text"       # contoh: "text"
LABEL_COLUMN = "sentimen"      # contoh: "sentiment"
DATE_COLUMN = "created_at"       # contoh: "created_at"

# Gunakan nilai lebih kecil ketika melakukan uji awal.
MAX_DOCUMENTS = 25000


In [ ]:
import pandas as pd
import torch

from src.preprocessing import (
    PreprocessConfig,
    TextPreprocessor,
    build_feature_bundle,
    load_text_dataset,
    save_feature_bundle,
    set_global_seed,
)

set_global_seed(42)
print("CUDA tersedia:", torch.cuda.is_available())


In [ ]:
df, columns = load_text_dataset(
    DATA_PATH,
    text_column=TEXT_COLUMN,
    label_column=LABEL_COLUMN,
    date_column=DATE_COLUMN,
)

if MAX_DOCUMENTS is not None:
    df = df.sample(
        n=min(MAX_DOCUMENTS, len(df)),
        random_state=42,
    ).reset_index(drop=True)

print(columns)
display(df.head())
print("Jumlah dokumen:", len(df))


In [ ]:
config = PreprocessConfig(
    text_column=columns["text_column"],
    label_column=columns["label_column"],
    date_column=columns["date_column"],
    min_df=3,
    max_df=0.95,
    max_features=12000,
    ngram_range=(1, 2),
    document_dim=256,
    random_state=42,
)

preprocessor = TextPreprocessor(config)
raw_texts = df[columns["text_column"]].astype(str).tolist()
processed_texts = preprocessor.transform(raw_texts)

valid_mask = [bool(text.strip()) for text in processed_texts]
df = df.loc[valid_mask].reset_index(drop=True)
raw_texts = [text for text, valid in zip(raw_texts, valid_mask) if valid]
processed_texts = [
    text for text, valid in zip(processed_texts, valid_mask) if valid
]

print("Dokumen setelah preprocessing:", len(processed_texts))
display(
    pd.DataFrame(
        {"raw": raw_texts[:5], "processed": processed_texts[:5]}
    )
)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

bundle = build_feature_bundle(
    raw_texts=raw_texts,
    processed_texts=processed_texts,
    config=config,
    semantic_embeddings=None,
    device=device,
)
save_feature_bundle(bundle, FEATURE_DIR)

# Simpan dataset lengkap agar label dan tanggal tetap tersedia.
df.to_parquet(CHECKPOINTS_DIR / "dataset_metadata.parquet", index=False)
pd.Series(columns).to_json(
    CHECKPOINTS_DIR / "resolved_columns.json",
    force_ascii=False,
    indent=2,
)

print("Bow:", bundle["bow"].shape)
print("TF-IDF:", bundle["tfidf"].shape)
print("Semantic:", bundle["semantic_embeddings"].shape)
print("Document:", bundle["document_embeddings"].shape)
print("Artefak tersimpan di:", FEATURE_DIR)


In [ ]:
# Pemeriksaan vocabulary
print(bundle["vocabulary"][:50])
